In [1]:
!pip install firebase-admin
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime  # Untuk konversi timestamp ke human-readable
import pandas as pd


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Konfigurasi database sumber
source_cred = credentials.Certificate("D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json")
source_app = firebase_admin.initialize_app(source_cred, {
  'databaseURL': 'https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/'
})

In [35]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Input tanggal dan waktu dalam format string
start_readable_date = "16-04-2026 16:41:00"
end_readable_date = "16-04-2026 20:05:00"

# 1. Tentukan zona waktu yang diinginkan
jakarta_tz = ZoneInfo("Asia/Jakarta")

# 2. Ubah string menjadi objek datetime yang "naive" (tanpa timezone)
naive_start_dt = datetime.strptime(start_readable_date, "%d-%m-%Y %H:%M:%S")
naive_end_dt = datetime.strptime(end_readable_date, "%d-%m-%Y %H:%M:%S")

# 3. Jadikan objek datetime "aware" dengan melokalkannya ke zona waktu Jakarta
aware_start_dt = naive_start_dt.replace(tzinfo=jakarta_tz)
aware_end_dt = naive_end_dt.replace(tzinfo=jakarta_tz)

# 4. Sekarang konversi ke Unix timestamp. Hasilnya akan akurat berdasarkan WIB.
start_timestamp = int(aware_start_dt.timestamp())
end_timestamp = int(aware_end_dt.timestamp())

# --- Verifikasi ---
print(f"String Asli       : {start_readable_date}")
print(f"Waktu (Aware WIB)  : {aware_start_dt}")
print(f"Unix Timestamp     : {start_timestamp}")
print("-" * 30)
print(f"String Asli       : {end_readable_date}")
print(f"Waktu (Aware WIB)  : {aware_end_dt}")
print(f"Unix Timestamp     : {end_timestamp}")

String Asli       : 16-04-2026 16:41:00
Waktu (Aware WIB)  : 2026-04-16 16:41:00+07:00
Unix Timestamp     : 1776332460
------------------------------
String Asli       : 16-04-2026 20:05:00
Waktu (Aware WIB)  : 2026-04-16 20:05:00+07:00
Unix Timestamp     : 1776344700


In [36]:
# Referensi ke data sumber
source_ref = db.reference('/auto_weather_stat/id-03/data')

# Mengambil data dari database sumber
source_data = source_ref.get()

# Mengubah data menjadi DataFrame pandas
if source_data:
    # Mengubah data menjadi DataFrame
    df = pd.DataFrame.from_dict(source_data, orient='index')
    
    # Jika 'timestamp' sudah ada, jangan pindahkan indeks ke kolom
    if 'timestamp' not in df.columns:
        df.index.name = 'timestamp'  # Mengatur nama indeks
        df.reset_index(inplace=True)  # Memindahkan indeks menjadi kolom biasa
    
    print(df.head())  # Menampilkan data untuk verifikasi
else:
    print("Tidak ada data di databse sumber.")

              dew humidity pressure temperature   timestamp  volt  humi  pres  \
1702296831  23.29    89.28  1010.72       25.18  1702296831  4.08   NaN   NaN   
1702296891  23.29    89.38  1010.75       25.16  1702296891  4.08   NaN   NaN   
1702296951  23.31    89.58  1010.76       25.14  1702296951  4.08   NaN   NaN   
1702297011  23.35    89.86  1010.82       25.13  1702297011  4.08   NaN   NaN   
1702297071  23.32    89.85  1010.82       25.10  1702297071  4.08   NaN   NaN   

            temp  rainfall  rainrate  
1702296831   NaN       NaN       NaN  
1702296891   NaN       NaN       NaN  
1702296951   NaN       NaN       NaN  
1702297011   NaN       NaN       NaN  
1702297071   NaN       NaN       NaN  


In [57]:
df.tail(10)

,dew,humidity,pressure,temperature,timestamp,volt,rainfall,rainrate
1776349025,26.33701,90.81,1010.39,27.98,1776349025,4.09,NaN,NaN
1776349085,26.28871,90.71,1010.56,27.95,1776349085,4.1,NaN,NaN
1776349150,26.21610,90.48,1010.44,27.92,1776349150,4.09,NaN,NaN
1776349206,26.22679,90.59,1010.40,27.91,1776349206,4.1,NaN,NaN
1776349266,26.22198,90.67,1010.42,27.89,1776349266,4.09,NaN,NaN
1776349326,26.22011,90.66,1010.42,27.89,1776349326,4.09,NaN,NaN
1776349386,26.13967,90.23,1010.46,27.89,1776349386,4.09,NaN,NaN
1776349446,26.08943,90.12,1010.59,27.86,1776349446,4.09,NaN,NaN
1776349506,26.12230,90.19,1010.48,27.88,1776349506,4.09,NaN,NaN
1776349566,26.17900,90.44,1010.47,27.89,1776349566,4.09,NaN,NaN


In [38]:
df = df.drop(['humi', 'temp', 'pres'], axis=1)

In [48]:
df.head(10)

,dew,humidity,pressure,temperature,timestamp,volt,rainfall,rainrate
1702296831,23.29,89.28,1011.72,25.18,1702296831,4.08,NaN,NaN
1702296891,23.29,89.38,1011.75,25.16,1702296891,4.08,NaN,NaN
1702296951,23.31,89.58,1011.76,25.14,1702296951,4.08,NaN,NaN
1702297011,23.35,89.86,1011.82,25.13,1702297011,4.08,NaN,NaN
1702297071,23.32,89.85,1011.82,25.10,1702297071,4.08,NaN,NaN
1702297131,23.29,89.81,1011.85,25.08,1702297131,4.08,NaN,NaN
1702297190,23.27,89.87,1011.84,25.05,1702297190,4.08,NaN,NaN
1702297251,23.26,89.88,1011.88,25.04,1702297251,4.09,NaN,NaN
1702297311,23.26,90.00,1011.86,25.02,1702297311,4.08,NaN,NaN
1702297371,23.28,90.27,1011.85,24.99,1702297371,4.08,NaN,NaN


In [49]:
df['dew'] = pd.to_numeric(df['dew'], errors='coerce')
df['humidity'] = pd.to_numeric(df['humidity'], errors='coerce')
df['pressure'] = pd.to_numeric(df['pressure'], errors='coerce')
df['temperature'] = pd.to_numeric(df['temperature'], errors='coerce')


In [50]:
df['pressure'] = df['pressure'].add(1)

In [51]:
df.head(10)

,dew,humidity,pressure,temperature,timestamp,volt,rainfall,rainrate
1702296831,23.29,89.28,1012.72,25.18,1702296831,4.08,NaN,NaN
1702296891,23.29,89.38,1012.75,25.16,1702296891,4.08,NaN,NaN
1702296951,23.31,89.58,1012.76,25.14,1702296951,4.08,NaN,NaN
1702297011,23.35,89.86,1012.82,25.13,1702297011,4.08,NaN,NaN
1702297071,23.32,89.85,1012.82,25.10,1702297071,4.08,NaN,NaN
1702297131,23.29,89.81,1012.85,25.08,1702297131,4.08,NaN,NaN
1702297190,23.27,89.87,1012.84,25.05,1702297190,4.08,NaN,NaN
1702297251,23.26,89.88,1012.88,25.04,1702297251,4.09,NaN,NaN
1702297311,23.26,90.00,1012.86,25.02,1702297311,4.08,NaN,NaN
1702297371,23.28,90.27,1012.85,24.99,1702297371,4.08,NaN,NaN


In [52]:
filtered_data = df

In [53]:
# Filter data berdasarkan rentang waktu dengan validasi kunci
filtered_data = {
    key: value
    for key, value in source_data.items()
    if key.isdigit() and start_timestamp <= int(key) <= end_timestamp
}

In [54]:
# Konfigurasi database tujuan
dest_cred = credentials.Certificate("D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json")
dest_app = firebase_admin.initialize_app(dest_cred, {
    'databaseURL': 'https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/'
}, name='destination')

ValueError: Firebase app named "destination" already exists. This means you called initialize_app() more than once with the same app name as the second argument. Make sure you provide a unique name every time you call initialize_app().

In [55]:
# Referensi ke data tujuan
dest_ref = db.reference('/auto_weather_stat/id-05/data', app=dest_app)

In [56]:
# Memindahkan data yang telah difilter ke database tujuan
if filtered_data:
    dest_ref.update(filtered_data)
    print("Data berhasil dipindahkan ke database tujuan.")
else:
    print("Tidak ada data dalam rentang waktu yang ditentukan.")

Data berhasil dipindahkan ke database tujuan.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=200afad5-8991-4e20-8a55-8751c7aff3b5' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>